# Memory Agent

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(model='gpt-4o-mini',temperature=0)

In [ ]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content:str=Field(description='The main content of the memory.For example: User expressed interest in learning about French.')
    
class MemoryCollection(BaseModel):
    memories:list[Memory]=Field(description='a list of memories about user')

In [ ]:
from trustcall import create_extractor

# Create the extractor
trustcall_extractor=create_extractor(
    model,
    tools=[Memory],
    tool_choice='Memory',
    enable_inserts=True    
)

# Inspect the tool calls made by Trustcall
class Spy:
    # Collect information about the tool calls made by the extractor.
    def __init__(self):
        self.called_tools=[]
    def __call__(self,run):
        q=[run]
        while q:
            r=q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type=='chat_model':
                self.called_tools.append(r.outputs['generations'][0][0]['message']['kwargs']['tool_calls'])

# Initialize the spy
spy=Spy()

# Add the spy as a listener
trustcall_extractor_see_all_tool_calls=trustcall_extractor.with_listeners(on_end=spy)